In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# OUTPUT FOLDERS

base_folder = "data/processed/paper24_fx"
tables_folder = f"{base_folder}/tables"
plots_folder = f"{base_folder}/plots"
timeseries_folder = f"{base_folder}/timeseries"

for folder in [base_folder, tables_folder, plots_folder, timeseries_folder]:
    os.makedirs(folder, exist_ok=True)

# LOAD DATA

ohlc = pd.read_csv(
    "fx_ohlc_2010_2025.csv",
    parse_dates=["Date"]
).sort_values("Date").reset_index(drop=True)

# FX pairs in your OHLC file

pair_prefixes = ["CADUSD", "CHFUSD", "EURUSD", "GBPUSD", "JPYUSD", "SEKUSD"]

# Detect DXY prefix in OHLC
dxy_prefix = None
for candidate in ["DXY", "DX=F", "DX-Y.NYB"]:
    if f"{candidate}_Close" in ohlc.columns:
        dxy_prefix = candidate
        break

if dxy_prefix is None:
    raise ValueError("Could not find DXY OHLC columns in fx_ohlc_2010_2025.csv")

print("Using DXY OHLC prefix:", dxy_prefix)

Using DXY OHLC prefix: DXY


In [2]:
for p in pair_prefixes:
    # raw sub-period returns
    ohlc[f"{p}_overnight_raw"] = ohlc[f"{p}_Open"] / ohlc[f"{p}_Close"].shift(1) - 1
    ohlc[f"{p}_daytime_raw"] = ohlc[f"{p}_Close"] / ohlc[f"{p}_Open"] - 1
    ohlc[f"{p}_cc_raw"] = ohlc[f"{p}_Close"].pct_change()

    # USD-aligned returns (invert XXXUSD)
    ohlc[f"{p}_overnight"] = -ohlc[f"{p}_overnight_raw"]
    ohlc[f"{p}_daytime"] = -ohlc[f"{p}_daytime_raw"]
    ohlc[f"{p}_cc"] = -ohlc[f"{p}_cc_raw"]

# DXY benchmark close-to-close
ohlc["DXY_cc"] = ohlc[f"{dxy_prefix}_Close"].pct_change().fillna(0)

ohlc.head()

C:\Users\Araj7\AppData\Local\Temp\ipykernel_6676\2068582894.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ohlc[f"{p}_cc_raw"] = ohlc[f"{p}_Close"].pct_change()
C:\Users\Araj7\AppData\Local\Temp\ipykernel_6676\2068582894.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ohlc[f"{p}_cc_raw"] = ohlc[f"{p}_Close"].pct_change()
C:\Users\Araj7\AppData\Local\Temp\ipykernel_6676\2068582894.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fi

,Date,EURUSD_Open,EURUSD_High,EURUSD_Low,EURUSD_Close,GBPUSD_Open,GBPUSD_High,GBPUSD_Low,GBPUSD_Close,CADUSD_Open,...,JPYUSD_overnight,JPYUSD_daytime,JPYUSD_cc,SEKUSD_overnight_raw,SEKUSD_daytime_raw,SEKUSD_cc_raw,SEKUSD_overnight,SEKUSD_daytime,SEKUSD_cc,DXY_cc
0,2010-12-01,1.299495,1.313456,1.297707,1.299207,1.557147,1.564798,1.555307,1.556808,0.975229,...,NaN,0.000143,NaN,NaN,-0.001011,NaN,NaN,0.001011,NaN,0.000000
1,2010-12-02,1.312560,1.321196,1.306506,1.312491,1.561061,1.566171,1.551518,1.561036,0.982415,...,0.005136,0.000154,0.005290,0.007617,-0.000402,0.007212,-0.007617,0.000402,-0.007212,-0.005080
2,2010-12-03,1.321301,1.337667,1.319505,1.321598,1.558992,1.571388,1.558312,1.559090,0.996711,...,-0.004574,0.000107,-0.004466,0.009934,0.000551,0.010490,-0.009934,-0.000551,-0.010490,-0.011457
3,2010-12-06,1.341706,1.341976,1.325047,1.341598,1.576839,1.577013,1.566097,1.576790,0.997606,...,-0.011828,0.000121,-0.011706,0.017094,-0.001001,0.016076,-0.017094,0.001001,-0.016076,0.002394
4,2010-12-07,1.329045,1.339800,1.328374,1.328692,1.571117,1.582404,1.570746,1.570870,0.994233,...,-0.001331,-0.000000,-0.001331,-0.008582,-0.000146,-0.008727,0.008582,0.000146,0.008727,0.003645


In [3]:
strategy_map = {
    1:  ("Long",     "Cash"),
    2:  ("Short",    "Cash"),
    3:  ("Cash",     "Long"),
    4:  ("Cash",     "Short"),
    5:  ("Long",     "Long"),
    6:  ("Short",    "Short"),
    7:  ("Short",    "Long"),
    8:  ("Long",     "Short"),
    9:  ("Cash",     "Inertia"),
    10: ("Cash",     "Reversal"),
    11: ("Inertia",  "Cash"),
    12: ("Reversal", "Cash"),
    13: ("Inertia",  "Inertia"),
    14: ("Inertia",  "Reversal"),
    15: ("Reversal", "Inertia"),
    16: ("Reversal", "Reversal"),
    17: ("Long",     "Inertia"),
    18: ("Long",     "Reversal"),
    19: ("Short",    "Inertia"),
    20: ("Short",    "Reversal"),
    21: ("Inertia",  "Long"),
    22: ("Reversal", "Long"),
    23: ("Inertia",  "Short"),
    24: ("Reversal", "Short"),
}

In [4]:
def signal_to_position(rule, signal_return):
    """
    rule in {Cash, Long, Short, Inertia, Reversal}
    signal_return is the relevant preceding sub-period return.
    """
    if rule == "Cash":
        return 0
    if rule == "Long":
        return 1
    if rule == "Short":
        return -1

    # Dynamic rules: non-negative => +1, negative => -1
    if pd.isna(signal_return):
        return 0

    base = 1 if signal_return >= 0 else -1

    if rule == "Inertia":
        return base
    if rule == "Reversal":
        return -base

    raise ValueError(f"Unknown rule: {rule}")


def compute_sharpe(return_series, annualization=252):
    r = pd.Series(return_series)
    r = pd.to_numeric(r, errors="coerce").dropna()

    if len(r) < 2:
        return np.nan

    std = r.std(ddof=1)
    if std == 0 or pd.isna(std):
        return np.nan

    return (r.mean() / std) * np.sqrt(annualization)


def compute_metrics_from_daily(daily_returns):
    daily_returns = pd.Series(pd.to_numeric(daily_returns, errors="coerce")).fillna(0)

    portfolio_value = 100 * (1 + daily_returns).cumprod()
    roll_max = portfolio_value.cummax()
    drawdown = portfolio_value / roll_max - 1

    return {
        "final_balance": float(portfolio_value.iloc[-1]),
        "sharpe": float(compute_sharpe(daily_returns, annualization=252)),
        "max_drawdown": float(drawdown.min()),
        "volatility": float(daily_returns.std(ddof=1) * np.sqrt(252))
    }

In [5]:
def run_strategy_for_pair(df, pair_prefix, overnight_rule, daytime_rule, cost_bps=0):
    """
    Returns:
      subperiod_df: sub-period-level results
      daily_df: daily aggregated results
      trade_count: number of trade episodes
    """
    dates = df["Date"].reset_index(drop=True)

    overnight_returns = df[f"{pair_prefix}_overnight"].reset_index(drop=True)
    daytime_returns = df[f"{pair_prefix}_daytime"].reset_index(drop=True)

    # previous daytime signal for overnight
    prev_daytime_signal = daytime_returns.shift(1)

    # current overnight signal for same-day daytime
    current_overnight_signal = overnight_returns.copy()

    subperiod_rows = []

    current_position = 0
    trade_count = 0
    cost_rate = cost_bps / 10000.0

    for i in range(len(df)):
        date_i = dates.iloc[i]

        # -------------------------
        # OVERNIGHT SUB-PERIOD
        # -------------------------
        pos_o = signal_to_position(overnight_rule, prev_daytime_signal.iloc[i])
        ret_o = overnight_returns.iloc[i]

        if pos_o != 0 and pos_o != current_position:
            trade_count += 1
            trade_cost_o = cost_rate
        else:
            trade_cost_o = 0.0

        pnl_o = 0.0 if pd.isna(ret_o) else pos_o * ret_o - trade_cost_o
        current_position = pos_o

        subperiod_rows.append({
            "Date": date_i,
            "Subperiod": "Overnight",
            "Position": pos_o,
            "RawReturn": ret_o,
            "NetReturn": pnl_o
        })

        # -------------------------
        # DAYTIME SUB-PERIOD
        # -------------------------
        pos_d = signal_to_position(daytime_rule, current_overnight_signal.iloc[i])
        ret_d = daytime_returns.iloc[i]

        if pos_d != 0 and pos_d != current_position:
            trade_count += 1
            trade_cost_d = cost_rate
        else:
            trade_cost_d = 0.0

        pnl_d = 0.0 if pd.isna(ret_d) else pos_d * ret_d - trade_cost_d
        current_position = pos_d

        subperiod_rows.append({
            "Date": date_i,
            "Subperiod": "Daytime",
            "Position": pos_d,
            "RawReturn": ret_d,
            "NetReturn": pnl_d
        })

    subperiod_df = pd.DataFrame(subperiod_rows)

    # aggregate 2 sub-periods into a daily return
    daily_df = (
        subperiod_df
        .pivot(index="Date", columns="Subperiod", values="NetReturn")
        .reset_index()
    )

    if "Overnight" not in daily_df.columns:
        daily_df["Overnight"] = 0.0
    if "Daytime" not in daily_df.columns:
        daily_df["Daytime"] = 0.0

    daily_df["DailyReturn"] = (1 + daily_df["Overnight"].fillna(0)) * (1 + daily_df["Daytime"].fillna(0)) - 1

    return subperiod_df, daily_df, trade_count

In [6]:
cost_scenarios = [0, 1, 2]

final_balance_tables = {}
sharpe_tables = {}
drawdown_tables = {}
vol_tables = {}
trade_count_tables = {}

all_summary_rows = []

for cost_bps in cost_scenarios:
    final_balance_table = pd.DataFrame(index=range(1, 25), columns=pair_prefixes)
    sharpe_table = pd.DataFrame(index=range(1, 25), columns=pair_prefixes)
    drawdown_table = pd.DataFrame(index=range(1, 25), columns=pair_prefixes)
    vol_table = pd.DataFrame(index=range(1, 25), columns=pair_prefixes)
    trade_count_table = pd.DataFrame(index=range(1, 25), columns=pair_prefixes)

    for strat_num, (overnight_rule, daytime_rule) in strategy_map.items():
        for p in pair_prefixes:
            pair_folder = f"{timeseries_folder}/{p}/cost_{cost_bps}bps"
            os.makedirs(pair_folder, exist_ok=True)

            sub_df, daily_df, trade_count = run_strategy_for_pair(
                ohlc, p, overnight_rule, daytime_rule, cost_bps=cost_bps
            )

            metrics = compute_metrics_from_daily(daily_df["DailyReturn"])

            final_balance_table.loc[strat_num, p] = metrics["final_balance"]
            sharpe_table.loc[strat_num, p] = metrics["sharpe"]
            drawdown_table.loc[strat_num, p] = metrics["max_drawdown"]
            vol_table.loc[strat_num, p] = metrics["volatility"]
            trade_count_table.loc[strat_num, p] = trade_count

            # save daily timeseries
            out_df = daily_df.copy()
            out_df["StrategyNum"] = strat_num
            out_df["OvernightRule"] = overnight_rule
            out_df["DaytimeRule"] = daytime_rule
            out_df.to_csv(
                f"{pair_folder}/strategy_{strat_num:02d}_{overnight_rule}_{daytime_rule}.csv",
                index=False
            )

            all_summary_rows.append({
                "CostBps": cost_bps,
                "StrategyNum": strat_num,
                "OvernightRule": overnight_rule,
                "DaytimeRule": daytime_rule,
                "Pair": p,
                "FinalBalance": metrics["final_balance"],
                "Sharpe": metrics["sharpe"],
                "MaxDrawdown": metrics["max_drawdown"],
                "Volatility": metrics["volatility"],
                "TradeCount": trade_count
            })

    # add rule columns
    meta_df = pd.DataFrame(
        [{"StrategyNum": k, "OvernightRule": v[0], "DaytimeRule": v[1]} for k, v in strategy_map.items()]
    ).set_index("StrategyNum")

    final_balance_table = meta_df.join(final_balance_table)
    sharpe_table = meta_df.join(sharpe_table)
    drawdown_table = meta_df.join(drawdown_table)
    vol_table = meta_df.join(vol_table)
    trade_count_table = meta_df.join(trade_count_table)

    final_balance_tables[cost_bps] = final_balance_table
    sharpe_tables[cost_bps] = sharpe_table
    drawdown_tables[cost_bps] = drawdown_table
    vol_tables[cost_bps] = vol_table
    trade_count_tables[cost_bps] = trade_count_table

    final_balance_table.to_csv(f"{tables_folder}/final_balances_{cost_bps}bps.csv")
    sharpe_table.to_csv(f"{tables_folder}/sharpe_{cost_bps}bps.csv")
    drawdown_table.to_csv(f"{tables_folder}/max_drawdown_{cost_bps}bps.csv")
    vol_table.to_csv(f"{tables_folder}/volatility_{cost_bps}bps.csv")
    trade_count_table.to_csv(f"{tables_folder}/trade_counts_{cost_bps}bps.csv")

all_summary = pd.DataFrame(all_summary_rows)
all_summary.to_csv(f"{tables_folder}/all_strategy_pair_summary.csv", index=False)

all_summary.head()

,CostBps,StrategyNum,OvernightRule,DaytimeRule,Pair,FinalBalance,Sharpe,MaxDrawdown,Volatility,TradeCount
0,0,1,Long,Cash,CADUSD,127.636613,0.253499,-0.176288,0.071883,3932
1,0,1,Long,Cash,CHFUSD,74.157348,-0.138965,-0.281717,0.100400,3932
2,0,1,Long,Cash,EURUSD,105.650116,0.083949,-0.198381,0.082174,3932
3,0,1,Long,Cash,GBPUSD,108.352865,0.102559,-0.228158,0.086521,3932
4,0,1,Long,Cash,JPYUSD,169.181972,0.414610,-0.206905,0.091348,3932


In [7]:
dxy_daily = ohlc[["Date", "DXY_cc"]].copy()
dxy_daily["DXY_cc"] = pd.to_numeric(dxy_daily["DXY_cc"], errors="coerce").fillna(0)

dxy_metrics = compute_metrics_from_daily(dxy_daily["DXY_cc"])

dxy_bh_summary = pd.DataFrame([{
    "Benchmark": "DXY Buy & Hold",
    "FinalBalance": dxy_metrics["final_balance"],
    "Sharpe": dxy_metrics["sharpe"],
    "MaxDrawdown": dxy_metrics["max_drawdown"],
    "Volatility": dxy_metrics["volatility"]
}])

dxy_bh_summary.to_csv(f"{tables_folder}/dxy_buyhold_summary.csv", index=False)
dxy_bh_summary

,Benchmark,FinalBalance,Sharpe,MaxDrawdown,Volatility
0,DXY Buy & Hold,121.719736,0.217867,-0.153186,0.068627


In [8]:
avg_rows = []

for cost_bps in cost_scenarios:
    temp = all_summary[all_summary["CostBps"] == cost_bps].copy()

    avg_table = (
        temp.groupby(["StrategyNum", "OvernightRule", "DaytimeRule"], as_index=False)
        .agg(
            AvgFinalBalance=("FinalBalance", "mean"),
            AvgSharpe=("Sharpe", "mean"),
            AvgMaxDrawdown=("MaxDrawdown", "mean"),
            AvgVolatility=("Volatility", "mean"),
            AvgTradeCount=("TradeCount", "mean"),
        )
        .sort_values("AvgFinalBalance", ascending=False)
        .reset_index(drop=True)
    )

    avg_table.to_csv(f"{tables_folder}/average_across_pairs_{cost_bps}bps.csv", index=False)
    avg_rows.append(avg_table.assign(CostBps=cost_bps))

avg_all = pd.concat(avg_rows, ignore_index=True)
avg_all.head(10)

,StrategyNum,OvernightRule,DaytimeRule,AvgFinalBalance,AvgSharpe,AvgMaxDrawdown,AvgVolatility,AvgTradeCount,CostBps
0,8,Long,Short,120.548939,0.169132,-0.229105,0.090442,7864.000000,0
1,18,Long,Reversal,117.736189,0.148632,-0.221518,0.090092,4019.333333,0
2,1,Long,Cash,116.646055,0.142020,-0.225620,0.090012,3932.000000,0
3,17,Long,Inertia,115.536488,0.134977,-0.231209,0.090291,3845.666667,0
4,24,Reversal,Short,113.601506,0.130050,-0.292853,0.090452,3147.000000,0
5,5,Long,Long,112.854409,0.114352,-0.232626,0.089941,1.000000,0
6,16,Reversal,Reversal,110.353022,0.109482,-0.299806,0.090012,3913.666667,0
7,12,Reversal,Cash,109.245137,0.102843,-0.303423,0.090014,3924.333333,0
8,15,Reversal,Inertia,108.121606,0.095989,-0.308455,0.090375,3936.000000,0
9,22,Reversal,Long,105.047497,0.075276,-0.317476,0.089936,4717.000000,0


In [9]:
for cost_bps in cost_scenarios:
    avg_table = pd.read_csv(f"{tables_folder}/average_across_pairs_{cost_bps}bps.csv")
    top10 = avg_table.head(10).copy()

    top10["Label"] = top10["StrategyNum"].astype(str) + ": " + top10["OvernightRule"] + " / " + top10["DaytimeRule"]

    # Final balance
    plt.figure(figsize=(12, 6))
    plt.bar(top10["Label"], top10["AvgFinalBalance"])
    plt.title(f"Top 10 FX Strategies by Avg Final Balance ({cost_bps} bps)")
    plt.xlabel("Strategy")
    plt.ylabel("Average Final Balance ($)")
    plt.ylim(bottom=0)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{plots_folder}/top10_avg_final_balance_{cost_bps}bps.png", dpi=300)
    plt.close()

    # Sharpe
    plt.figure(figsize=(12, 6))
    plt.bar(top10["Label"], top10["AvgSharpe"])
    plt.title(f"Top 10 FX Strategies by Avg Sharpe ({cost_bps} bps)")
    plt.xlabel("Strategy")
    plt.ylabel("Average Sharpe")
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{plots_folder}/top10_avg_sharpe_{cost_bps}bps.png", dpi=300)
    plt.close()

    # Drawdown magnitude
    plt.figure(figsize=(12, 6))
    plt.bar(top10["Label"], top10["AvgMaxDrawdown"].abs())
    plt.title(f"Top 10 FX Strategies by Avg Max Drawdown Magnitude ({cost_bps} bps)")
    plt.xlabel("Strategy")
    plt.ylabel("Average Drawdown Magnitude")
    plt.ylim(bottom=0)
    plt.xticks(rotation=45, ha="right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{plots_folder}/top10_avg_drawdown_{cost_bps}bps.png", dpi=300)
    plt.close()

print("Plots saved.")

Plots saved.


In [10]:
focus_strats = [5, 18]

focus = all_summary[all_summary["StrategyNum"].isin(focus_strats)].copy()

for cost_bps in cost_scenarios:
    temp = focus[focus["CostBps"] == cost_bps].copy()

    pivot_fb = temp.pivot_table(
        index=["StrategyNum", "OvernightRule", "DaytimeRule"],
        columns="Pair",
        values="FinalBalance"
    )

    pivot_fb.to_csv(f"{tables_folder}/focus_strategy_5_18_final_balance_{cost_bps}bps.csv")

    plt.figure(figsize=(12, 6))
    labels = [
        f"{int(idx[0])}: {idx[1]}/{idx[2]}"
        for idx in pivot_fb.index
    ]

    x = np.arange(len(pair_prefixes))
    width = 0.35

    plt.bar(x - width/2, pivot_fb.iloc[0].values, width=width, label=labels[0])
    plt.bar(x + width/2, pivot_fb.iloc[1].values, width=width, label=labels[1])

    plt.xticks(x, pair_prefixes)
    plt.ylabel("Final Balance ($)")
    plt.xlabel("FX Pair")
    plt.title(f"Strategy #5 vs #18 by FX Pair ({cost_bps} bps)")
    plt.ylim(bottom=0)
    plt.grid(axis="y", alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{plots_folder}/strategy5_vs_18_{cost_bps}bps.png", dpi=300)
    plt.close()

In [11]:
import os
import numpy as np
import pandas as pd

# make sure output folder exists
os.makedirs(tables_folder, exist_ok=True)

def build_paper_style_final_balance_table(all_summary_df, cost_bps, pair_cols):
    """
    Creates a full 24 x pair table like the paper image.
    """
    temp = all_summary_df[all_summary_df["CostBps"] == cost_bps].copy()

    pivot_fb = temp.pivot_table(
        index=["StrategyNum", "OvernightRule", "DaytimeRule"],
        columns="Pair",
        values="FinalBalance"
    ).reset_index()

    # order by strategy number
    pivot_fb = pivot_fb.sort_values("StrategyNum").reset_index(drop=True)

    # round numeric columns for display
    for c in pair_cols:
        pivot_fb[c] = pd.to_numeric(pivot_fb[c], errors="coerce").round(2)

    return pivot_fb


paper_table_0bps = build_paper_style_final_balance_table(all_summary, 0, pair_prefixes)
paper_table_1bps = build_paper_style_final_balance_table(all_summary, 1, pair_prefixes)
paper_table_2bps = build_paper_style_final_balance_table(all_summary, 2, pair_prefixes)

paper_table_0bps.to_csv(f"{tables_folder}/paper_style_final_balance_0bps.csv", index=False)
paper_table_1bps.to_csv(f"{tables_folder}/paper_style_final_balance_1bps.csv", index=False)
paper_table_2bps.to_csv(f"{tables_folder}/paper_style_final_balance_2bps.csv", index=False)

paper_table_0bps.head(24)

Pair,StrategyNum,OvernightRule,DaytimeRule,CADUSD,CHFUSD,EURUSD,GBPUSD,JPYUSD,SEKUSD
0,1,Long,Cash,127.64,74.16,105.65,108.35,169.18,114.90
1,2,Short,Cash,72.28,115.14,85.19,82.12,51.89,72.62
2,3,Cash,Long,96.00,91.91,96.29,96.05,98.94,98.57
3,4,Cash,Short,104.16,108.79,103.84,104.10,101.05,101.14
4,5,Long,Long,122.53,68.16,101.73,104.07,167.39,113.25
5,6,Short,Short,75.28,125.26,88.46,85.49,52.43,73.44
6,7,Short,Long,69.39,105.82,82.02,78.87,51.34,71.57
7,8,Long,Short,132.95,80.68,109.71,112.80,170.96,116.20
8,9,Cash,Inertia,98.52,99.08,101.37,98.42,100.80,95.50
9,10,Cash,Reversal,101.49,100.92,98.64,101.59,99.19,104.39


In [12]:
def style_paper_table(df, pair_cols):
    """
    Styles the dataframe like the paper:
    - Strategy #5 row blue
    - green if value > strategy #5 for that pair
    - red if value < strategy #5 for that pair
    """
    benchmark_row = df[df["StrategyNum"] == 5].iloc[0]

    def color_cell(val, row_strategy_num, col_name):
        if col_name not in pair_cols:
            return ""

        bench_val = benchmark_row[col_name]

        # Strategy #5 row = blue
        if row_strategy_num == 5:
            return "background-color: #c9d6ff;"   # light blue

        if pd.isna(val) or pd.isna(bench_val):
            return ""

        if val > bench_val:
            return "background-color: #cfeccf;"   # light green
        elif val < bench_val:
            return "background-color: #f6caca;"   # light red
        else:
            return "background-color: #eeeeee;"   # neutral gray

    styled = df.style

    for col in df.columns:
        if col in pair_cols:
            styled = styled.apply(
                lambda s, col_name=col: [
                    color_cell(v, df.iloc[i]["StrategyNum"], col_name)
                    for i, v in enumerate(s)
                ],
                subset=[col]
            )

    return styled

In [13]:
def export_styled_excel(df, output_path, pair_cols):
    """
    Save styled paper table to Excel with colors.
    """
    benchmark_row = df[df["StrategyNum"] == 5].iloc[0]

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="FinalBalances", index=False)

        workbook = writer.book
        ws = writer.sheets["FinalBalances"]

        from openpyxl.styles import PatternFill, Font, Alignment

        blue_fill = PatternFill(fill_type="solid", start_color="C9D6FF", end_color="C9D6FF")
        green_fill = PatternFill(fill_type="solid", start_color="CFECCF", end_color="CFECCF")
        red_fill = PatternFill(fill_type="solid", start_color="F6CACA", end_color="F6CACA")
        gray_fill = PatternFill(fill_type="solid", start_color="EEEEEE", end_color="EEEEEE")

        # header formatting
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal="center", vertical="center")

        # map column names to Excel positions
        col_idx = {cell.value: i + 1 for i, cell in enumerate(ws[1])}

        for row_num in range(2, ws.max_row + 1):
            strategy_num = ws.cell(row=row_num, column=col_idx["StrategyNum"]).value

            for pair in pair_cols:
                c = ws.cell(row=row_num, column=col_idx[pair])
                val = c.value
                bench_val = benchmark_row[pair]

                if strategy_num == 5:
                    c.fill = blue_fill
                else:
                    if val is None or bench_val is None:
                        continue
                    if val > bench_val:
                        c.fill = green_fill
                    elif val < bench_val:
                        c.fill = red_fill
                    else:
                        c.fill = gray_fill

        # widen columns
        for column_cells in ws.columns:
            max_length = 0
            column_letter = column_cells[0].column_letter
            for cell in column_cells:
                try:
                    max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            ws.column_dimensions[column_letter].width = max_length + 2

In [14]:
export_styled_excel(
    paper_table_0bps,
    f"{tables_folder}/paper_style_final_balance_0bps.xlsx",
    pair_prefixes
)

export_styled_excel(
    paper_table_1bps,
    f"{tables_folder}/paper_style_final_balance_1bps.xlsx",
    pair_prefixes
)

export_styled_excel(
    paper_table_2bps,
    f"{tables_folder}/paper_style_final_balance_2bps.xlsx",
    pair_prefixes
)

print("Styled Excel tables saved.")

Styled Excel tables saved.


In [15]:
styled_0bps = style_paper_table(paper_table_0bps, pair_prefixes)
styled_1bps = style_paper_table(paper_table_1bps, pair_prefixes)
styled_2bps = style_paper_table(paper_table_2bps, pair_prefixes)

styled_0bps

Pair,StrategyNum,OvernightRule,DaytimeRule,CADUSD,CHFUSD,EURUSD,GBPUSD,JPYUSD,SEKUSD
0,1,Long,Cash,127.640000,74.160000,105.650000,108.350000,169.180000,114.900000
1,2,Short,Cash,72.280000,115.140000,85.190000,82.120000,51.890000,72.620000
2,3,Cash,Long,96.000000,91.910000,96.290000,96.050000,98.940000,98.570000
3,4,Cash,Short,104.160000,108.790000,103.840000,104.100000,101.050000,101.140000
4,5,Long,Long,122.530000,68.160000,101.730000,104.070000,167.390000,113.250000
5,6,Short,Short,75.280000,125.260000,88.460000,85.490000,52.430000,73.440000
6,7,Short,Long,69.390000,105.820000,82.020000,78.870000,51.340000,71.570000
7,8,Long,Short,132.950000,80.680000,109.710000,112.800000,170.960000,116.200000
8,9,Cash,Inertia,98.520000,99.080000,101.370000,98.420000,100.800000,95.500000
9,10,Cash,Reversal,101.490000,100.920000,98.640000,101.590000,99.190000,104.390000


In [20]:
def best_strategy_per_pair(all_summary_df, cost_bps):
    temp = all_summary_df[all_summary_df["CostBps"] == cost_bps].copy()

    rows = []
    for p in pair_prefixes:
        sub = temp[temp["Pair"] == p].copy()
        best_row = sub.loc[sub["FinalBalance"].idxmax()]

        rows.append({
            "Pair": p,
            "BestStrategyNum": int(best_row["StrategyNum"]),
            "OvernightRule": best_row["OvernightRule"],
            "DaytimeRule": best_row["DaytimeRule"],
            "BestFinalBalance": round(best_row["FinalBalance"], 2),
            "Sharpe": round(best_row["Sharpe"], 4) if pd.notna(best_row["Sharpe"]) else np.nan,
            "MaxDrawdown": round(best_row["MaxDrawdown"], 4),
            "TradeCount": int(best_row["TradeCount"])
        })

    return pd.DataFrame(rows)


best_pair_0bps = best_strategy_per_pair(all_summary, 0)
best_pair_1bps = best_strategy_per_pair(all_summary, 1)
best_pair_2bps = best_strategy_per_pair(all_summary, 2)

best_pair_0bps.to_csv(f"{tables_folder}/best_strategy_per_pair_0bps.csv", index=False)
best_pair_1bps.to_csv(f"{tables_folder}/best_strategy_per_pair_1bps.csv", index=False)
best_pair_2bps.to_csv(f"{tables_folder}/best_strategy_per_pair_2bps.csv", index=False)

best_pair_0bps

,Pair,BestStrategyNum,OvernightRule,DaytimeRule,BestFinalBalance,Sharpe,MaxDrawdown,TradeCount
0,CADUSD,24,Reversal,Short,144.20,0.3620,-0.1890,2992
1,CHFUSD,6,Short,Short,125.26,0.1928,-0.3050,1
2,EURUSD,24,Reversal,Short,145.94,0.3356,-0.2375,2909
3,GBPUSD,8,Long,Short,112.80,0.1323,-0.2285,7864
4,JPYUSD,8,Long,Short,170.96,0.4220,-0.2023,7864
5,SEKUSD,18,Long,Reversal,119.94,0.1622,-0.2387,3941


In [21]:
def overall_best_strategy(all_summary_df, cost_bps):
    temp = all_summary_df[all_summary_df["CostBps"] == cost_bps].copy()
    best_row = temp.loc[temp["FinalBalance"].idxmax()]

    return pd.DataFrame([{
        "CostBps": cost_bps,
        "Pair": best_row["Pair"],
        "StrategyNum": int(best_row["StrategyNum"]),
        "OvernightRule": best_row["OvernightRule"],
        "DaytimeRule": best_row["DaytimeRule"],
        "FinalBalance": round(best_row["FinalBalance"], 2),
        "Sharpe": round(best_row["Sharpe"], 4) if pd.notna(best_row["Sharpe"]) else np.nan,
        "MaxDrawdown": round(best_row["MaxDrawdown"], 4),
        "TradeCount": int(best_row["TradeCount"])
    }])


overall_best_0bps = overall_best_strategy(all_summary, 0)
overall_best_1bps = overall_best_strategy(all_summary, 1)
overall_best_2bps = overall_best_strategy(all_summary, 2)

overall_best = pd.concat([overall_best_0bps, overall_best_1bps, overall_best_2bps], ignore_index=True)
overall_best.to_csv(f"{tables_folder}/overall_best_strategy_each_cost.csv", index=False)

overall_best

,CostBps,Pair,StrategyNum,OvernightRule,DaytimeRule,FinalBalance,Sharpe,MaxDrawdown,TradeCount
0,0,JPYUSD,8,Long,Short,170.96,0.4220,-0.2023,7864
1,1,JPYUSD,5,Long,Long,167.39,0.4067,-0.2156,1
2,2,JPYUSD,5,Long,Long,167.39,0.4067,-0.2156,1


In [22]:
top10_overall = (
    all_summary.sort_values(["CostBps", "FinalBalance"], ascending=[True, False])
    .groupby("CostBps", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

top10_overall.to_csv(f"{tables_folder}/top10_overall_max_returns_by_cost.csv", index=False)
top10_overall

,CostBps,StrategyNum,OvernightRule,DaytimeRule,Pair,FinalBalance,Sharpe,MaxDrawdown,Volatility,TradeCount
0,0,8,Long,Short,JPYUSD,170.957756,0.421961,-0.202286,0.091341,7864
1,0,17,Long,Inertia,JPYUSD,170.534944,0.420003,-0.202961,0.091402,3739
2,0,1,Long,Cash,JPYUSD,169.181972,0.414610,-0.206905,0.091348,3932
3,0,18,Long,Reversal,JPYUSD,167.810000,0.408656,-0.211247,0.091418,4126
4,0,5,Long,Long,JPYUSD,167.394956,0.406708,-0.215584,0.091479,1
5,0,24,Reversal,Short,EURUSD,145.939805,0.335598,-0.237480,0.082274,2909
6,0,24,Reversal,Short,CADUSD,144.204085,0.362001,-0.189018,0.071961,2992
7,0,15,Reversal,Inertia,EURUSD,142.458303,0.316941,-0.249164,0.082220,3957
8,0,12,Reversal,Cash,EURUSD,140.539527,0.306535,-0.248756,0.082160,3925
9,0,16,Reversal,Reversal,CADUSD,140.512454,0.338678,-0.186635,0.072021,3971


 DXY for 24 


In [16]:
# -----------------------------------
# PREP DXY OVERNIGHT / DAYTIME RETURNS
# -----------------------------------
ohlc["DXY_overnight"] = ohlc[f"{dxy_prefix}_Open"] / ohlc[f"{dxy_prefix}_Close"].shift(1) - 1
ohlc["DXY_daytime"] = ohlc[f"{dxy_prefix}_Close"] / ohlc[f"{dxy_prefix}_Open"] - 1
ohlc["DXY_cc"] = ohlc[f"{dxy_prefix}_Close"].pct_change().fillna(0)

def run_strategy_for_dxy(df, overnight_rule, daytime_rule, cost_bps=0):
    dates = df["Date"].reset_index(drop=True)

    overnight_returns = df["DXY_overnight"].reset_index(drop=True)
    daytime_returns = df["DXY_daytime"].reset_index(drop=True)

    prev_daytime_signal = daytime_returns.shift(1)
    current_overnight_signal = overnight_returns.copy()

    subperiod_rows = []

    current_position = 0
    trade_count = 0
    cost_rate = cost_bps / 10000.0

    for i in range(len(df)):
        date_i = dates.iloc[i]

        # Overnight
        pos_o = signal_to_position(overnight_rule, prev_daytime_signal.iloc[i])
        ret_o = overnight_returns.iloc[i]

        if pos_o != 0 and pos_o != current_position:
            trade_count += 1
            trade_cost_o = cost_rate
        else:
            trade_cost_o = 0.0

        pnl_o = 0.0 if pd.isna(ret_o) else pos_o * ret_o - trade_cost_o
        current_position = pos_o

        subperiod_rows.append({
            "Date": date_i,
            "Subperiod": "Overnight",
            "Position": pos_o,
            "RawReturn": ret_o,
            "NetReturn": pnl_o
        })

        # Daytime
        pos_d = signal_to_position(daytime_rule, current_overnight_signal.iloc[i])
        ret_d = daytime_returns.iloc[i]

        if pos_d != 0 and pos_d != current_position:
            trade_count += 1
            trade_cost_d = cost_rate
        else:
            trade_cost_d = 0.0

        pnl_d = 0.0 if pd.isna(ret_d) else pos_d * ret_d - trade_cost_d
        current_position = pos_d

        subperiod_rows.append({
            "Date": date_i,
            "Subperiod": "Daytime",
            "Position": pos_d,
            "RawReturn": ret_d,
            "NetReturn": pnl_d
        })

    subperiod_df = pd.DataFrame(subperiod_rows)

    daily_df = (
        subperiod_df
        .pivot(index="Date", columns="Subperiod", values="NetReturn")
        .reset_index()
    )

    if "Overnight" not in daily_df.columns:
        daily_df["Overnight"] = 0.0
    if "Daytime" not in daily_df.columns:
        daily_df["Daytime"] = 0.0

    daily_df["DailyReturn"] = (1 + daily_df["Overnight"].fillna(0)) * (1 + daily_df["Daytime"].fillna(0)) - 1

    return subperiod_df, daily_df, trade_count

C:\Users\Araj7\AppData\Local\Temp\ipykernel_6676\399385864.py:6: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  ohlc["DXY_cc"] = ohlc[f"{dxy_prefix}_Close"].pct_change().fillna(0)


In [17]:
dxy_summary_rows = []

for cost_bps in [0, 1, 2]:
    dxy_table = []

    dxy_cost_folder = f"{timeseries_folder}/DXY/cost_{cost_bps}bps"
    os.makedirs(dxy_cost_folder, exist_ok=True)

    for strat_num, (overnight_rule, daytime_rule) in strategy_map.items():
        sub_df, daily_df, trade_count = run_strategy_for_dxy(
            ohlc, overnight_rule, daytime_rule, cost_bps=cost_bps
        )

        metrics = compute_metrics_from_daily(daily_df["DailyReturn"])

        daily_out = daily_df.copy()
        daily_out["StrategyNum"] = strat_num
        daily_out["OvernightRule"] = overnight_rule
        daily_out["DaytimeRule"] = daytime_rule

        daily_out.to_csv(
            f"{dxy_cost_folder}/strategy_{strat_num:02d}_{overnight_rule}_{daytime_rule}.csv",
            index=False
        )

        row = {
            "StrategyNum": strat_num,
            "OvernightRule": overnight_rule,
            "DaytimeRule": daytime_rule,
            "FinalBalance": metrics["final_balance"],
            "Sharpe": metrics["sharpe"],
            "MaxDrawdown": metrics["max_drawdown"],
            "Volatility": metrics["volatility"],
            "TradeCount": trade_count,
            "CostBps": cost_bps
        }

        dxy_table.append(row)
        dxy_summary_rows.append(row)

    dxy_table_df = pd.DataFrame(dxy_table).sort_values("StrategyNum").reset_index(drop=True)
    dxy_table_df.to_csv(f"{tables_folder}/paper_style_dxy_{cost_bps}bps.csv", index=False)

dxy_all_summary = pd.DataFrame(dxy_summary_rows)
dxy_all_summary.to_csv(f"{tables_folder}/all_dxy_strategy_summary.csv", index=False)

dxy_all_summary.head()

,StrategyNum,OvernightRule,DaytimeRule,FinalBalance,Sharpe,MaxDrawdown,Volatility,TradeCount,CostBps
0,1,Long,Cash,61.021625,-1.431541,-0.411805,0.021944,3932,0
1,2,Short,Cash,162.640029,1.431541,-0.053063,0.021944,3932,0
2,3,Cash,Long,196.013922,0.680610,-0.123945,0.066641,3932,0
3,4,Cash,Short,47.595932,-0.680610,-0.603188,0.066641,3932,0
4,5,Long,Long,119.610881,0.202743,-0.142794,0.068017,1,0


In [18]:
best_dxy_rows = []

for cost_bps in [0, 1, 2]:
    temp = dxy_all_summary[dxy_all_summary["CostBps"] == cost_bps].copy()
    best_row = temp.loc[temp["FinalBalance"].idxmax()]

    best_dxy_rows.append({
        "CostBps": cost_bps,
        "BestStrategyNum": int(best_row["StrategyNum"]),
        "OvernightRule": best_row["OvernightRule"],
        "DaytimeRule": best_row["DaytimeRule"],
        "FinalBalance": round(best_row["FinalBalance"], 2),
        "Sharpe": round(best_row["Sharpe"], 4) if pd.notna(best_row["Sharpe"]) else np.nan,
        "MaxDrawdown": round(best_row["MaxDrawdown"], 4),
        "TradeCount": int(best_row["TradeCount"])
    })

best_dxy_table = pd.DataFrame(best_dxy_rows)
best_dxy_table.to_csv(f"{tables_folder}/best_dxy_strategy_each_cost.csv", index=False)

best_dxy_table

,CostBps,BestStrategyNum,OvernightRule,DaytimeRule,FinalBalance,Sharpe,MaxDrawdown,TradeCount
0,0,7,Short,Long,318.80,1.0642,-0.1305,7864
1,1,20,Short,Reversal,171.96,0.5235,-0.3640,3894
2,2,5,Long,Long,119.61,0.2027,-0.1428,1


In [23]:
overall_fx_best = pd.read_csv(f"{tables_folder}/overall_best_strategy_each_cost.csv")
best_dxy = pd.read_csv(f"{tables_folder}/best_dxy_strategy_each_cost.csv")

comparison_best = overall_fx_best.merge(best_dxy, on="CostBps", suffixes=("_FX", "_DXY"))
comparison_best.to_csv(f"{tables_folder}/best_fx_vs_best_dxy_by_cost.csv", index=False)

comparison_best

,CostBps,Pair,StrategyNum,OvernightRule_FX,DaytimeRule_FX,FinalBalance_FX,Sharpe_FX,MaxDrawdown_FX,TradeCount_FX,BestStrategyNum,OvernightRule_DXY,DaytimeRule_DXY,FinalBalance_DXY,Sharpe_DXY,MaxDrawdown_DXY,TradeCount_DXY
0,0,JPYUSD,8,Long,Short,170.96,0.4220,-0.2023,7864,7,Short,Long,318.80,1.0642,-0.1305,7864
1,1,JPYUSD,5,Long,Long,167.39,0.4067,-0.2156,1,20,Short,Reversal,171.96,0.5235,-0.3640,3894
2,2,JPYUSD,5,Long,Long,167.39,0.4067,-0.2156,1,5,Long,Long,119.61,0.2027,-0.1428,1


In [24]:
import os
import numpy as np
import pandas as pd

# make sure folder exists
os.makedirs(tables_folder, exist_ok=True)

all_asset_cols = ["CADUSD", "CHFUSD", "EURUSD", "GBPUSD", "JPYUSD", "SEKUSD", "DXY"]

def build_combined_paper_table(all_summary_df, dxy_summary_df, cost_bps):
    # FX part
    fx_temp = all_summary_df[all_summary_df["CostBps"] == cost_bps].copy()

    fx_pivot = fx_temp.pivot_table(
        index=["StrategyNum", "OvernightRule", "DaytimeRule"],
        columns="Pair",
        values="FinalBalance"
    )

    # DXY part
    dxy_temp = dxy_summary_df[dxy_summary_df["CostBps"] == cost_bps].copy()

    dxy_pivot = dxy_temp.pivot_table(
        index=["StrategyNum", "OvernightRule", "DaytimeRule"],
        values="FinalBalance"
    )

    dxy_pivot.columns = ["DXY"]

    # combine
    combined = fx_pivot.join(dxy_pivot, how="outer").reset_index()
    combined = combined.sort_values("StrategyNum").reset_index(drop=True)

    # ensure all columns exist
    for col in all_asset_cols:
        if col not in combined.columns:
            combined[col] = np.nan

    combined = combined[["StrategyNum", "OvernightRule", "DaytimeRule"] + all_asset_cols]

    # round for display
    for col in all_asset_cols:
        combined[col] = pd.to_numeric(combined[col], errors="coerce").round(2)

    return combined


paper_full_0bps = build_combined_paper_table(all_summary, dxy_all_summary, 0)
paper_full_1bps = build_combined_paper_table(all_summary, dxy_all_summary, 1)
paper_full_2bps = build_combined_paper_table(all_summary, dxy_all_summary, 2)

paper_full_0bps.to_csv(f"{tables_folder}/paper_style_full_fx_dxy_0bps.csv", index=False)
paper_full_1bps.to_csv(f"{tables_folder}/paper_style_full_fx_dxy_1bps.csv", index=False)
paper_full_2bps.to_csv(f"{tables_folder}/paper_style_full_fx_dxy_2bps.csv", index=False)

paper_full_0bps.head(24)

,StrategyNum,OvernightRule,DaytimeRule,CADUSD,CHFUSD,EURUSD,GBPUSD,JPYUSD,SEKUSD,DXY
0,1,Long,Cash,127.64,74.16,105.65,108.35,169.18,114.90,61.02
1,2,Short,Cash,72.28,115.14,85.19,82.12,51.89,72.62,162.64
2,3,Cash,Long,96.00,91.91,96.29,96.05,98.94,98.57,196.01
3,4,Cash,Short,104.16,108.79,103.84,104.10,101.05,101.14,47.60
4,5,Long,Long,122.53,68.16,101.73,104.07,167.39,113.25,119.61
5,6,Short,Short,75.28,125.26,88.46,85.49,52.43,73.44,77.41
6,7,Short,Long,69.39,105.82,82.02,78.87,51.34,71.57,318.80
7,8,Long,Short,132.95,80.68,109.71,112.80,170.96,116.20,29.04
8,9,Cash,Inertia,98.52,99.08,101.37,98.42,100.80,95.50,61.20
9,10,Cash,Reversal,101.49,100.92,98.64,101.59,99.19,104.39,152.83


In [25]:
from openpyxl.styles import PatternFill, Font, Alignment

def export_full_styled_excel(df, output_path, asset_cols):
    benchmark_row = df[df["StrategyNum"] == 5].iloc[0]

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="FinalBalances", index=False)

        ws = writer.sheets["FinalBalances"]

        blue_fill = PatternFill(fill_type="solid", start_color="C9D6FF", end_color="C9D6FF")
        green_fill = PatternFill(fill_type="solid", start_color="CFECCF", end_color="CFECCF")
        red_fill = PatternFill(fill_type="solid", start_color="F6CACA", end_color="F6CACA")
        gray_fill = PatternFill(fill_type="solid", start_color="EEEEEE", end_color="EEEEEE")

        # header style
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(horizontal="center", vertical="center")

        # column map
        col_idx = {cell.value: i + 1 for i, cell in enumerate(ws[1])}

        for row_num in range(2, ws.max_row + 1):
            strategy_num = ws.cell(row=row_num, column=col_idx["StrategyNum"]).value

            for asset in asset_cols:
                c = ws.cell(row=row_num, column=col_idx[asset])
                val = c.value
                bench_val = benchmark_row[asset]

                if strategy_num == 5:
                    c.fill = blue_fill
                else:
                    if val is None or pd.isna(val) or bench_val is None or pd.isna(bench_val):
                        continue
                    if val > bench_val:
                        c.fill = green_fill
                    elif val < bench_val:
                        c.fill = red_fill
                    else:
                        c.fill = gray_fill

        # widen columns
        for column_cells in ws.columns:
            max_length = 0
            column_letter = column_cells[0].column_letter
            for cell in column_cells:
                try:
                    max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            ws.column_dimensions[column_letter].width = max_length + 2

In [26]:
export_full_styled_excel(
    paper_full_0bps,
    f"{tables_folder}/paper_style_full_fx_dxy_0bps.xlsx",
    all_asset_cols
)

export_full_styled_excel(
    paper_full_1bps,
    f"{tables_folder}/paper_style_full_fx_dxy_1bps.xlsx",
    all_asset_cols
)

export_full_styled_excel(
    paper_full_2bps,
    f"{tables_folder}/paper_style_full_fx_dxy_2bps.xlsx",
    all_asset_cols
)

print("Full FX + DXY styled Excel tables saved.")

Full FX + DXY styled Excel tables saved.


In [ ]:
def find_overall_best_from_full_table(df, asset_cols):
    best_val = -np.inf
    best_info = None

    for _, row in df.iterrows():
        strat_num = row["StrategyNum"]
        overnight_rule = row["OvernightRule"]
        daytime_rule = row["DaytimeRule"]

        for asset in asset_cols:
            val = row[asset]
            if pd.notna(val) and val > best_val:
                best_val = val
                best_info = {
                    "StrategyNum": int(strat_num),
                    "OvernightRule": overnight_rule,
                    "DaytimeRule": daytime_rule,
                    "Asset": asset,
                    "FinalBalance": round(val, 2)
                }

    return pd.DataFrame([best_info])


best_full_0bps = find_overall_best_from_full_table(paper_full_0bps, all_asset_cols)
best_full_1bps = find_overall_best_from_full_table(paper_full_1bps, all_asset_cols)
best_full_2bps = find_overall_best_from_full_table(paper_full_2bps, all_asset_cols)

best_full_all = pd.concat([
    best_full_0bps.assign(CostBps=0),
    best_full_1bps.assign(CostBps=1),
    best_full_2bps.assign(CostBps=2)
], ignore_index=True)

best_full_all.to_csv(f"{tables_folder}/overall_best_fx_dxy_each_cost.csv", index=False)
best_full_all

,StrategyNum,OvernightRule,DaytimeRule,Asset,FinalBalance,CostBps
0,7,Short,Long,DXY,318.80,0
1,20,Short,Reversal,DXY,171.96,1
2,5,Long,Long,JPYUSD,167.39,2


: 